# 00 — Ground Truth Extraction

**Propósito:** Extraer datos CROSS-VALIDADOS desde dos fuentes independientes (MT5 broker + Dukascopy)
y construir el dataset de ground truth en UTC explícito.

**Regla de oro:** El broker MT5 es la fuente de verdad (es lo que vio el operador).  
Dukascopy es la fuente de referencia externa para detectar anomalías.

---
| Item | Valor |
|------|-------|
| Cuenta | MEX Atlantic 921339 (investor read-only) |
| Símbolo broker | `XAUUSD..` (dos puntos) |
| Servidor MT5 | GMT+3 (confirmado) |
| Almacenamiento | Todo en UTC |
| Trades reales | 30 (2026-03-19 → 2026-04-27) |

In [ ]:
# ── Celda 1: Imports y configuración ──────────────────────────────────────────
import sys
import logging
from pathlib import Path

# Asegurar que la raíz del proyecto esté en sys.path
PROJECT_ROOT = Path('..').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# Configurar logging para ver todo el output de los módulos
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(name)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S',
)

from v2.config.settings import (
    PERIOD_START_UTC, PERIOD_END_UTC,
    TRADES_PERIOD_START_UTC, TRADES_PERIOD_END_UTC,
    TIMEFRAMES, SYMBOL_BROKER, SYMBOL_STANDARD,
    MT5_GMT_OFFSET_HOURS, DATA_RAW_MT5, DATA_RAW_DUKA,
    DATA_GROUND_TRUTH, DATA_REPORTS,
    TOTAL_TRADES,
)

print('=== CONFIGURACIÓN ===')
print(f'  Símbolo broker : {SYMBOL_BROKER}')
print(f'  Símbolo estándar: {SYMBOL_STANDARD}')
print(f'  GMT offset servidor MT5: +{MT5_GMT_OFFSET_HOURS}h (UTC+3)')
print(f'  Periodo análisis: {PERIOD_START_UTC} → {PERIOD_END_UTC}')
print(f'  Periodo trades: {TRADES_PERIOD_START_UTC} → {TRADES_PERIOD_END_UTC}')
print(f'  Timeframes: {TIMEFRAMES}')
print(f'  Trades esperados: {TOTAL_TRADES}')
print('Configuración OK ✓')

In [ ]:
# ── Celda 2: Extraer historial de trades desde MT5 ────────────────────────────
from v2.src.data.mt5_extractor import extract_trade_history

print('Extrayendo trades desde MT5...')
trades = extract_trade_history(save_parquet=True)

print(f'\n=== TRADES EXTRAÍDOS ===')
print(f'  Total trades: {len(trades)}')
print(f'  Esperados: {TOTAL_TRADES}')

if len(trades) == TOTAL_TRADES:
    print('  ✓ Conteo correcto')
else:
    print(f'  ⚠ DIFERENCIA: esperados {TOTAL_TRADES}, obtenidos {len(trades)}')

print(f'\n  Periodo: {trades["time_open_utc"].min()} → {trades["time_open_utc"].max()}')
print(f'  Profit total: ${trades["profit"].sum():.2f}')
print(f'  BUY: {(trades["type"]=="BUY").sum()} | SELL: {(trades["type"]=="SELL").sum()}')
print(f'  Win rate: {(trades["profit"] > 0).mean():.1%}')
print(f'  Duración media: {trades["duration_minutes"].mean():.0f} min')

print('\n=== PRIMERAS 10 FILAS ===')
display(trades.head(10))

In [ ]:
# ── Celda 3: Extraer OHLCV desde MT5 (todos los timeframes) ───────────────────
from v2.src.data.mt5_extractor import extract_all_timeframes

print(f'Extrayendo OHLCV desde MT5 para {len(TIMEFRAMES)} timeframes...')
print(f'Periodo: {PERIOD_START_UTC} → {PERIOD_END_UTC}')
print()

mt5_ohlc = extract_all_timeframes(
    timeframes=TIMEFRAMES,
    start_utc=PERIOD_START_UTC,
    end_utc=PERIOD_END_UTC,
)

print('\n=== RESUMEN MT5 OHLCV ===')
for tf, df in mt5_ohlc.items():
    if df.empty:
        print(f'  {tf:5s}: ⚠ VACÍO')
    else:
        path = DATA_RAW_MT5 / f'ohlc_{tf}.parquet'
        size_kb = path.stat().st_size / 1024 if path.exists() else 0
        print(
            f'  {tf:5s}: {len(df):6,d} velas | '
            f'{df.index.min().strftime("%Y-%m-%d")} → {df.index.max().strftime("%Y-%m-%d")} | '
            f'{size_kb:.0f} KB'
        )

print(f'\n=== M15 (muestra 5 filas) ===')
if 'M15' in mt5_ohlc and not mt5_ohlc['M15'].empty:
    display(mt5_ohlc['M15'].head())

In [ ]:
# ── Celda 4: Extraer OHLCV desde Dukascopy ────────────────────────────────────
# Primero verificamos que la librería esté instalada
try:
    import dukascopy
    print(f'dukascopy-python instalado: v{dukascopy.__version__}')
except ImportError:
    print('⚠ dukascopy-python NO instalado.')
    print('Instalando...')
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'dukascopy-python', '-q'])
    print('Instalado. Reinicia el kernel si hay errores de import.')

from v2.src.data.dukascopy_extractor import extract_all_timeframes as duka_extract_all

# Excluimos D1 para Dukascopy — a veces tiene limitaciones en rangos largos
DUKA_TIMEFRAMES = [tf for tf in TIMEFRAMES if tf not in ['D1']]

print(f'\nDescargando Dukascopy para: {DUKA_TIMEFRAMES}')
print(f'Periodo: {PERIOD_START_UTC} → {PERIOD_END_UTC}')
print('(Puede tomar varios minutos por rate limiting)\n')

duka_ohlc = duka_extract_all(
    timeframes=DUKA_TIMEFRAMES,
    start_utc=PERIOD_START_UTC,
    end_utc=PERIOD_END_UTC,
)

print('\n=== RESUMEN DUKASCOPY OHLCV ===')
for tf, df in duka_ohlc.items():
    if df.empty:
        print(f'  {tf:5s}: ⚠ VACÍO')
    else:
        path = DATA_RAW_DUKA / f'ohlc_{tf}.parquet'
        size_kb = path.stat().st_size / 1024 if path.exists() else 0
        print(
            f'  {tf:5s}: {len(df):6,d} velas | '
            f'{df.index.min().strftime("%Y-%m-%d")} → {df.index.max().strftime("%Y-%m-%d")} | '
            f'{size_kb:.0f} KB'
        )

print(f'\n=== M15 Dukascopy (muestra 5 filas) ===')
if 'M15' in duka_ohlc and not duka_ohlc['M15'].empty:
    display(duka_ohlc['M15'].head())

In [ ]:
# ── Celda 5: Validación cross-fuente ──────────────────────────────────────────
from v2.src.data.data_validator import validate_ohlc_sources, report_validation_summary

VALIDATE_TFS = ['M5', 'M15', 'H1']
print(f'Validando MT5 vs Dukascopy para: {VALIDATE_TFS}')
print('=' * 60)

validation_results = {}
for tf in VALIDATE_TFS:
    print(f'\n--- {tf} ---')
    comparison, stats = validate_ohlc_sources(tf, save_report=True)
    validation_results[tf] = (comparison, stats)

    if not comparison.empty:
        print(f'Barras alineadas: {stats["n_aligned"]:,}')
        print(f'Dentro de tolerancia (≤1 pip): {stats["pct_within_tolerance"]:.1f}%')
        print(f'Diff close media: {stats["diff_close_mean_pips"]:.4f} pips')
        print(f'Diff close p95: {stats["diff_close_p95_pips"]:.4f} pips')
        print(f'Diff close max: {stats["diff_close_max_pips"]:.4f} pips')

        if stats['pct_within_tolerance'] >= 95:
            print('✓ VALIDACIÓN APROBADA (≥95% dentro de 1 pip)')
        elif stats['pct_within_tolerance'] >= 80:
            print('⚠ VALIDACIÓN MARGINAL (80-95% dentro de 1 pip)')
        else:
            print('✗ VALIDACIÓN FALLIDA (<80% dentro de 1 pip) — revisar zona horaria')

print('\n\n=== TABLA COMPARATIVA ===')
summary_rows = [
    {
        'TF': tf,
        'Barras MT5': validation_results[tf][1].get('n_mt5_bars', 0),
        'Barras Duka': validation_results[tf][1].get('n_duka_bars', 0),
        'Alineadas': validation_results[tf][1].get('n_aligned', 0),
        '% Dentro 1pip': f"{validation_results[tf][1].get('pct_within_tolerance', 0):.1f}%",
        'Diff close media (pips)': f"{validation_results[tf][1].get('diff_close_mean_pips', 0):.4f}",
        'Max diff (pips)': f"{validation_results[tf][1].get('max_diff_max_pips', 0):.4f}",
    }
    for tf in VALIDATE_TFS if validation_results[tf][1]
]
display(pd.DataFrame(summary_rows).set_index('TF'))

print(f'\nReportes HTML generados en: {DATA_REPORTS}')

In [ ]:
# ── Celda 6: Ground Truth Decision ────────────────────────────────────────────
# Fuente de verdad: MT5 broker (lo que vio el operador)
# Copia los datos MT5 a data/ground_truth/ documentando el nivel de desacuerdo
import shutil

DATA_GROUND_TRUTH.mkdir(parents=True, exist_ok=True)

print('=== DECISIÓN GROUND TRUTH ===')
print('Fuente elegida: MT5 broker (MEX Atlantic)')
print('Razón: el operador vio los precios de este broker, no de Dukascopy.')
print()

ground_truth_log = []

# Trades
trades_src = DATA_RAW_MT5 / 'trades.parquet'
if trades_src.exists():
    shutil.copy2(trades_src, DATA_GROUND_TRUTH / 'trades.parquet')
    print(f'✓ trades.parquet copiado ({trades_src.stat().st_size/1024:.1f} KB)')
else:
    print('⚠ trades.parquet no encontrado — ejecuta Celda 2 primero')

# OHLCV por timeframe
for tf in TIMEFRAMES:
    src = DATA_RAW_MT5 / f'ohlc_{tf}.parquet'
    if not src.exists():
        print(f'  {tf:5s}: ⚠ archivo MT5 no encontrado')
        continue

    # Calcular desacuerdo con Dukascopy si está disponible
    disagreement_info = {}
    if tf in validation_results and not validation_results[tf][0].empty:
        comp, stats = validation_results[tf]
        n_out = stats.get('n_outliers', 0)
        pct = stats.get('pct_within_tolerance', 0)
        disagreement_info = {'n_outliers': n_out, 'pct_within_tol': pct}
        note = f'{n_out} barras con diff >1pip ({100-pct:.1f}%)'
    else:
        note = 'sin validación cross-fuente'

    dst = DATA_GROUND_TRUTH / f'ohlc_{tf}.parquet'
    shutil.copy2(src, dst)
    size_kb = dst.stat().st_size / 1024
    print(f'  {tf:5s}: ✓ copiado ({size_kb:.0f} KB) | {note}')

    ground_truth_log.append({'timeframe': tf, 'size_kb': size_kb, **disagreement_info})

# Guardar log de decisión
if ground_truth_log:
    gt_log_df = pd.DataFrame(ground_truth_log)
    gt_log_df.to_parquet(DATA_GROUND_TRUTH / 'ground_truth_log.parquet', index=False)
    print(f'\nLog de decisión guardado en ground_truth/ground_truth_log.parquet')

print('\nGround truth listo ✓')

In [ ]:
# ── Celda 7: Resumen final ─────────────────────────────────────────────────────
print('=' * 60)
print('FASE 0 — RESUMEN FINAL')
print('=' * 60)

# 1. Trades
trades_path = DATA_GROUND_TRUTH / 'trades.parquet'
if trades_path.exists():
    t = pd.read_parquet(trades_path)
    print(f'\n[TRADES]')
    print(f'  Extraídos: {len(t)} (esperados: {TOTAL_TRADES})')
    print(f'  Periodo UTC: {t["time_open_utc"].min()} → {t["time_open_utc"].max()}')
    print(f'  Profit total: ${t["profit"].sum():.2f}')
    print(f'  Win rate: {(t["profit"]>0).mean():.1%}')
else:
    print('\n[TRADES] ⚠ No disponible')

# 2. OHLCV por timeframe
print(f'\n[OHLCV GROUND TRUTH]')
total_size_mb = 0
for tf in TIMEFRAMES:
    p = DATA_GROUND_TRUTH / f'ohlc_{tf}.parquet'
    if p.exists():
        df = pd.read_parquet(p)
        size_kb = p.stat().st_size / 1024
        total_size_mb += size_kb / 1024
        print(f'  {tf:5s}: {len(df):7,d} velas | {size_kb:.0f} KB')
    else:
        print(f'  {tf:5s}: ⚠ faltante')
print(f'  TOTAL: {total_size_mb:.1f} MB')

# 3. Validación cross-fuente
print(f'\n[VALIDACIÓN CROSS-FUENTE]')
for tf in VALIDATE_TFS:
    if tf in validation_results and validation_results[tf][1]:
        stats = validation_results[tf][1]
        pct = stats.get('pct_within_tolerance', 0)
        status = '✓' if pct >= 95 else ('⚠' if pct >= 80 else '✗')
        print(f'  {status} {tf:5s}: {pct:.1f}% barras dentro de 1 pip')
    else:
        print(f'  ? {tf:5s}: no validado')

# 4. Archivos generados
print(f'\n[ARCHIVOS GENERADOS]')
for folder, label in [
    (DATA_RAW_MT5, 'raw_mt5'),
    (DATA_RAW_DUKA, 'raw_dukascopy'),
    (DATA_GROUND_TRUTH, 'ground_truth'),
    (DATA_REPORTS, 'reports'),
]:
    if folder.exists():
        files = list(folder.glob('*'))
        total_kb = sum(f.stat().st_size for f in files if f.is_file()) / 1024
        print(f'  {label}: {len(files)} archivos | {total_kb:.0f} KB')
    else:
        print(f'  {label}: carpeta no encontrada')

print()
print('Fase 0 completada. Próximo paso: notebooks/01_eda.ipynb')
print('=' * 60)